# 4. Results Review: SHAP + FFA + Scenario Analysis

**Purpose:** Final workflow notebook for summarizing and reviewing completed model results. This replaces a dashboard notebook as the last analytical review step.

This notebook reads existing artifacts from Steps 6-9 and produces review tables and visualizations for:

1. Final model performance and availability.
2. SHAP global feature importance.
3. FFA / pseudo-causal feature importance.
4. Combined SHAP + FFA scenario interaction artifacts.
5. Patient-level explanation coverage.
6. DTW trajectory artifact availability.

Run this notebook after `3_model_train_shap_ffa.ipynb` and after optional DTW generation.

## Setup

Configure paths, cohorts, age bands, and plotting defaults. The notebook uses local artifacts created by the workflow and does not mutate pipeline outputs.

In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 180

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "py_helpers").exists():
    PROJECT_ROOT = Path("C:/Projects/cpic_time_to_event_analysis")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.constants import REQUIRED_COHORTS, PROJECT_SLUG, S3_BUCKET, age_band_to_fname
from py_helpers.event_density_utils import DENSITY_BINS, list_trained_density_bins, cohort_aggregate_final_model_has_artifacts

RESULTS_ROOT = PROJECT_ROOT / "10_analysis_results" / "visualizations" / "scenario"
RESULTS_REVIEW_ROOT = PROJECT_ROOT / "10_analysis_results" / "visualizations" / "results_review"
RESULTS_REVIEW_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_ROOT = PROJECT_ROOT / "6_final_model" / "outputs"
SHAP_ROOT = PROJECT_ROOT / "7_shap_analysis" / "outputs"
FFA_ROOT = PROJECT_ROOT / "8_ffa_analysis" / "outputs"
DTW_ROOT = PROJECT_ROOT / "9_dtw_analysis" / "outputs"


def save_review_figure(filename: str) -> Path:
    path = RESULTS_REVIEW_ROOT / filename
    plt.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"Saved review figure: {path.relative_to(PROJECT_ROOT)}")
    return path

print(f"Project root: {PROJECT_ROOT}")
print(f"Project slug: {PROJECT_SLUG}")
print(f"S3 bucket: {S3_BUCKET}")
print(f"Cohorts: {REQUIRED_COHORTS}")

## Artifact Inventory

This table answers the first review question: do we have the expected model, SHAP, FFA, combined scenario, and DTW artifacts for each cohort/age band/bin?

In [ ]:
def _scenario_dir(cohort, age_band, bin_name=None):
    base = RESULTS_ROOT / cohort / age_band_to_fname(age_band)
    return base / "bin_models" / bin_name if bin_name else base


def _shap_dir(cohort, age_band, bin_name=None):
    base = SHAP_ROOT / cohort / age_band_to_fname(age_band)
    return base / "bin_models" / bin_name if bin_name else base


def _ffa_dir(cohort, age_band, bin_name=None):
    base = FFA_ROOT / cohort / age_band_to_fname(age_band)
    return base / "bin_models" / bin_name if bin_name else base


def _artifact_row(cohort, age_band, bin_name=None):
    abf = age_band_to_fname(age_band)
    scenario_dir = _scenario_dir(cohort, age_band, bin_name)
    shap_dir = _shap_dir(cohort, age_band, bin_name)
    ffa_dir = _ffa_dir(cohort, age_band, bin_name)
    model_base = FINAL_MODEL_ROOT / cohort / abf
    if bin_name:
        model_base = model_base / "bin_models" / bin_name
    return {
        "cohort": cohort,
        "age_band": age_band,
        "bin": bin_name or "aggregate",
        "model_xgb": (model_base / "models" / "xgboost.joblib").exists() or (model_base / "models" / "xgboost_model.ubj").exists(),
        "model_catboost": (model_base / "models" / "catboost.joblib").exists() or (model_base / "models" / "catboost_model.cbm").exists(),
        "metrics": (model_base / f"{cohort}_{abf}_model_metrics_summary.csv").exists(),
        "shap_importance": (shap_dir / f"{cohort}_{abf}_shap_global_importance_xgboost.csv").exists(),
        "shap_sample": (shap_dir / f"{cohort}_{abf}_shap_sample_values_xgboost.parquet").exists(),
        "ffa_importance": (ffa_dir / "xgboost" / "feature_importance_axp.parquet").exists(),
        "ffa_explanations": (ffa_dir / "xgboost" / "axp_explanations.parquet").exists(),
        "combined_dashboard_json": (scenario_dir / "dashboard_data.json").exists(),
        "combined_importance": (scenario_dir / "combined_importance.csv").exists(),
        "patient_explanations": (scenario_dir / "patient_explanations.csv").exists(),
        "dtw_outputs_dir": (DTW_ROOT / cohort / abf).exists(),
        "scenario_dir": str(scenario_dir),
    }

rows = []
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        trained_bins = list_trained_density_bins(PROJECT_ROOT, cohort, age_band)
        if trained_bins:
            for bin_name in trained_bins:
                rows.append(_artifact_row(cohort, age_band, bin_name))
        if cohort_aggregate_final_model_has_artifacts(PROJECT_ROOT, cohort, age_band) or not trained_bins:
            rows.append(_artifact_row(cohort, age_band, None))

artifact_inventory = pd.DataFrame(rows)
display_cols = [
    "cohort", "age_band", "bin", "model_xgb", "model_catboost", "metrics",
    "shap_importance", "shap_sample", "ffa_importance", "ffa_explanations",
    "combined_dashboard_json", "combined_importance", "patient_explanations", "dtw_outputs_dir",
]
display(artifact_inventory[display_cols])

In [ ]:
if artifact_inventory.empty:
    print("No Step 6+ artifacts found yet. Run `3_model_train_shap_ffa.ipynb` first.")
else:
    bool_cols = [c for c in display_cols if c not in {"cohort", "age_band", "bin"}]
    coverage = artifact_inventory[bool_cols].mean().sort_values()
    ax = coverage.plot(kind="barh", figsize=(9, 5), color="#2563eb")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Fraction available")
    ax.set_title("Artifact availability across configured cohort / age-band / bin rows")
    for i, v in enumerate(coverage):
        ax.text(v + 0.02, i, f"{v:.0%}", va="center")
    plt.tight_layout()
    save_review_figure("artifact_availability.png")
    plt.show()

## Final Model Performance

Review Step 6 model metrics where available. The notebook searches aggregate and per-bin metric summaries.

In [ ]:
def load_model_metrics():
    frames = []
    for _, row in artifact_inventory.iterrows():
        cohort, age_band, bin_name = row["cohort"], row["age_band"], row["bin"]
        abf = age_band_to_fname(age_band)
        base = FINAL_MODEL_ROOT / cohort / abf
        if bin_name != "aggregate":
            base = base / "bin_models" / bin_name
        path = base / f"{cohort}_{abf}_model_metrics_summary.csv"
        if path.exists():
            df = pd.read_csv(path)
            df["cohort"] = cohort
            df["age_band"] = age_band
            df["bin"] = bin_name
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

model_metrics = load_model_metrics()
if model_metrics.empty:
    print("No model metrics found yet.")
else:
    display(model_metrics.head(20))

In [ ]:
if not model_metrics.empty:
    metric_candidates = [c for c in ["mean_pr_auc", "pr_auc", "auroc", "mean_auroc", "logloss", "mean_logloss"] if c in model_metrics.columns]
    if metric_candidates:
        metric = metric_candidates[0]
        plot_df = model_metrics.copy()
        if "model" not in plot_df.columns:
            plot_df["model"] = plot_df.get("selected_model", "model")
        plt.figure(figsize=(11, max(4, 0.35 * len(plot_df))))
        sns.barplot(data=plot_df, y="model", x=metric, hue="cohort", errorbar=None)
        plt.title(f"Model Performance Review: {metric}")
        plt.xlabel(metric)
        plt.ylabel("Model")
        plt.tight_layout()
        save_review_figure(f"model_performance_{metric}.png")
        plt.show()
    else:
        print("Metric summary found, but no standard metric columns were detected.")

## Combined SHAP + FFA Importance

This section reviews the interaction/scenario artifacts generated by `10_analysis_results/data_preparation/combine_shap_ffa_results.py`. These are the core results for scenario and pseudo-causal feature importance review.

In [ ]:
def load_combined_outputs():
    combined_frames = []
    consensus_records = []
    patient_summary = []
    dashboard_records = []
    for _, row in artifact_inventory.iterrows():
        cohort, age_band, bin_name = row["cohort"], row["age_band"], row["bin"]
        scenario_dir = Path(row["scenario_dir"])
        combined_path = scenario_dir / "combined_importance.csv"
        consensus_path = scenario_dir / "consensus_features.json"
        patient_path = scenario_dir / "patient_explanations.csv"
        dashboard_path = scenario_dir / "dashboard_data.json"
        label = {"cohort": cohort, "age_band": age_band, "bin": bin_name}
        if combined_path.exists():
            df = pd.read_csv(combined_path)
            for k, v in label.items():
                df[k] = v
            combined_frames.append(df)
        if consensus_path.exists():
            with open(consensus_path, "r", encoding="utf-8") as f:
                rec = json.load(f)
            rec.update(label)
            consensus_records.append(rec)
        if patient_path.exists():
            pdf = pd.read_csv(patient_path)
            patient_summary.append({
                **label,
                "n_patient_explanations": len(pdf),
                "n_with_consensus": int(pdf.get("consensus_features", pd.Series([], dtype=object)).astype(str).ne("[]").sum()) if len(pdf) else 0,
            })
        if dashboard_path.exists():
            with open(dashboard_path, "r", encoding="utf-8") as f:
                dash = json.load(f)
            dashboard_records.append({
                **label,
                "total_features": dash.get("summary", {}).get("total_features"),
                "top_feature": dash.get("summary", {}).get("top_feature"),
                "top_feature_importance": dash.get("summary", {}).get("top_feature_importance"),
            })
    return (
        pd.concat(combined_frames, ignore_index=True) if combined_frames else pd.DataFrame(),
        pd.DataFrame(consensus_records),
        pd.DataFrame(patient_summary),
        pd.DataFrame(dashboard_records),
    )

combined_importance, consensus_summary, patient_summary, dashboard_summary = load_combined_outputs()
print(f"Combined rows: {len(combined_importance):,}")
display(dashboard_summary)

In [ ]:
if combined_importance.empty:
    print("No combined SHAP+FFA outputs found yet. Run Step 8 in `3_model_train_shap_ffa.ipynb`.")
else:
    top = (combined_importance
           .sort_values("combined_importance", ascending=False)
           .groupby(["cohort", "age_band", "bin"], as_index=False)
           .head(15))
    n_panels = top[["cohort", "age_band", "bin"]].drop_duplicates().shape[0]
    g = sns.catplot(
        data=top,
        x="combined_importance",
        y="feature",
        col="cohort",
        row="age_band",
        hue="bin",
        kind="bar",
        sharey=False,
        height=4,
        aspect=1.4,
        errorbar=None,
    )
    g.fig.suptitle("Top Combined SHAP + FFA Feature Importances", y=1.02)
    g.set_axis_labels("Combined importance", "Feature")
    g.fig.savefig(RESULTS_REVIEW_ROOT / "top_combined_shap_ffa_importance.png", bbox_inches="tight", facecolor="white")
    print(f"Saved review figure: {(RESULTS_REVIEW_ROOT / 'top_combined_shap_ffa_importance.png').relative_to(PROJECT_ROOT)}")
    plt.show()

In [ ]:
if not combined_importance.empty:
    heat = combined_importance.copy()
    heat["panel"] = heat["cohort"] + " / " + heat["age_band"] + " / " + heat["bin"]
    top_features = (heat.groupby("feature")["combined_importance"].max()
                    .sort_values(ascending=False).head(30).index)
    mat = (heat[heat["feature"].isin(top_features)]
           .pivot_table(index="feature", columns="panel", values="combined_importance", aggfunc="max", fill_value=0)
           .loc[top_features])
    plt.figure(figsize=(max(9, 0.8 * mat.shape[1]), max(8, 0.28 * mat.shape[0])))
    sns.heatmap(mat, cmap="viridis", linewidths=0.2, linecolor="white")
    plt.title("Combined SHAP + FFA Importance Heatmap")
    plt.xlabel("Cohort / age band / bin")
    plt.ylabel("Feature")
    plt.tight_layout()
    save_review_figure("combined_shap_ffa_heatmap.png")
    plt.show()

## Consensus, Feature Types, and Patient Explanations

These checks follow the FFA analysis reference: consensus features indicate agreement between SHAP and FFA, feature-type mix helps verify whether the result is driven by drugs, diagnoses, procedures, or other model features, and patient explanations show how often global consensus features appear in individual explanations.

In [ ]:
if consensus_summary.empty:
    print("No consensus feature JSON files found yet.")
else:
    cols = [c for c in ["cohort", "age_band", "bin", "consensus_count", "shap_count", "ffa_count", "consensus_rate"] if c in consensus_summary.columns]
    display(consensus_summary[cols])
    plt.figure(figsize=(10, 4))
    sns.barplot(data=consensus_summary, x="age_band", y="consensus_count", hue="cohort", errorbar=None)
    plt.title("SHAP + FFA Consensus Feature Counts")
    plt.xlabel("Age band")
    plt.ylabel("Consensus features")
    plt.tight_layout()
    save_review_figure("consensus_feature_counts.png")
    plt.show()

In [ ]:
def feature_type(feature):
    feature = str(feature)
    if feature.startswith(("item_drug_", "drug_")):
        return "drug"
    if feature.startswith(("item_icd_", "icd_")):
        return "icd"
    if feature.startswith(("item_cpt_", "cpt_")):
        return "cpt"
    if feature.startswith("pgx_") or "pgx" in feature.lower():
        return "pgx"
    return "other"

if not combined_importance.empty:
    ft = combined_importance.copy()
    ft["feature_type"] = ft["feature"].map(feature_type)
    ft_counts = (ft.groupby(["cohort", "age_band", "bin", "feature_type"])
                   .size().reset_index(name="n_features"))
    display(ft_counts)
    pivot = ft_counts.pivot_table(index=["cohort", "age_band", "bin"], columns="feature_type", values="n_features", fill_value=0)
    pivot.plot(kind="bar", stacked=True, figsize=(11, 5), colormap="tab20")
    plt.title("Feature Type Mix in Combined Importance Outputs")
    plt.xlabel("Cohort / age band / bin")
    plt.ylabel("Feature count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_review_figure("combined_feature_type_mix.png")
    plt.show()

In [ ]:
if patient_summary.empty:
    print("No patient explanation summaries found yet.")
else:
    patient_summary["pct_with_consensus"] = np.where(
        patient_summary["n_patient_explanations"] > 0,
        patient_summary["n_with_consensus"] / patient_summary["n_patient_explanations"],
        0,
    )
    display(patient_summary)
    plt.figure(figsize=(10, 4))
    sns.barplot(data=patient_summary, x="age_band", y="pct_with_consensus", hue="cohort", errorbar=None)
    plt.title("Patient Explanations Containing Consensus Features")
    plt.xlabel("Age band")
    plt.ylabel("Percent with consensus feature")
    plt.gca().yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    plt.tight_layout()
    save_review_figure("patient_explanations_consensus.png")
    plt.show()

## SHAP vs FFA Agreement

The scatter below compares normalized SHAP importance and normalized FFA importance. Features in the upper-right are candidates for stronger review because both methods rank them highly.

In [ ]:
if not combined_importance.empty and {"shap_norm", "ffa_norm"}.issubset(combined_importance.columns):
    plot_df = combined_importance.copy()
    plot_df["feature_type"] = plot_df["feature"].map(feature_type)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=plot_df, x="shap_norm", y="ffa_norm", hue="feature_type", alpha=0.75)
    plt.title("SHAP vs FFA Normalized Importance")
    plt.xlabel("SHAP normalized importance")
    plt.ylabel("FFA normalized importance")
    plt.tight_layout()
    save_review_figure("shap_vs_ffa_agreement.png")
    plt.show()

    display(plot_df.sort_values("combined_importance", ascending=False)[
        ["cohort", "age_band", "bin", "feature", "feature_type", "shap_norm", "ffa_norm", "combined_importance"]
    ].head(30))

## Persist and Sync Results Artifacts

This cell writes review tables and saved figures under `10_analysis_results/visualizations/results_review/`, then syncs both the combined scenario artifacts and Results notebook visualizations to project-scoped S3:

- `s3://{S3_BUCKET}/gold/{PROJECT_SLUG}/analysis_visuals/scenario/`
- `s3://{S3_BUCKET}/gold/{PROJECT_SLUG}/analysis_visuals/results_review/`

In [ ]:
# Persist tabular review outputs locally for S3 sync.
review_tables = {
    "artifact_inventory.csv": artifact_inventory if "artifact_inventory" in globals() else pd.DataFrame(),
    "model_metrics.csv": model_metrics if "model_metrics" in globals() else pd.DataFrame(),
    "combined_importance.csv": combined_importance if "combined_importance" in globals() else pd.DataFrame(),
    "consensus_summary.csv": consensus_summary if "consensus_summary" in globals() else pd.DataFrame(),
    "patient_summary.csv": patient_summary if "patient_summary" in globals() else pd.DataFrame(),
    "dashboard_summary.csv": dashboard_summary if "dashboard_summary" in globals() else pd.DataFrame(),
}
for filename, df in review_tables.items():
    path = RESULTS_REVIEW_ROOT / filename
    df.to_csv(path, index=False)
    print(f"Saved review table: {path.relative_to(PROJECT_ROOT)}")

sync_script_dir = PROJECT_ROOT / "10_analysis_results" / "data_preparation"
if str(sync_script_dir) not in sys.path:
    sys.path.insert(0, str(sync_script_dir))
import sync_results_artifacts_s3

uploaded = sync_results_artifacts_s3.sync_results_artifacts(dry_run=False)
print(f"Uploaded/synced {len(uploaded)} results artifact(s) to project-scoped S3.")
if uploaded:
    print("Example S3 outputs:")
    for uri in uploaded[:10]:
        print(f"  {uri}")

## Final Summary

After running the notebook, use the generated tables and figures above to write the final interpretation. Focus on:

### Data Analysis Key Findings

- Which cohort/age-band/bin combinations have complete model, SHAP, FFA, combined scenario, and DTW artifacts?
- Which features are consistently important across SHAP and FFA?
- Are top features clinically plausible and free of target-definition leakage?
- Do patient explanations frequently include the consensus features?

### Insights or Next Steps

- Re-run missing workflow steps for incomplete artifact rows before final manuscript/result interpretation.
- Review top combined SHAP+FFA features against the Step 3b leakage filters and target-definition exclusions before presenting causal language.

## Optional EC2 Shutdown

Run this final cell only after the results review and S3 sync are complete. Set `SHUTDOWN_EC2 = True` to stop the current EC2 instance. This stops the instance; it does not terminate it.

In [ ]:
SHUTDOWN_EC2 = False  # Set to True only after confirming results are synced.

print("=" * 80)
print("Final Step: EC2 Instance Shutdown (Optional)")
print("=" * 80)

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)

    import os
    import shutil
    import subprocess

    try:
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5,
        )
        instance_id = result.stdout.strip()

        if instance_id:
            print(f"Instance ID: {instance_id}")

            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                for path in [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws",
                ]:
                    if os.path.exists(path):
                        aws_cmd = path
                        break

            if not aws_cmd:
                print("\nWarning: AWS CLI not found. Cannot stop instance.")
                print("Install AWS CLI or ensure it is in your PATH.")
            else:
                shutdown_cmd = [
                    aws_cmd,
                    "ec2",
                    "stop-instances",
                    "--instance-ids",
                    instance_id,
                ]
                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(shutdown_cmd, capture_output=True, text=True)

                if result.returncode == 0:
                    print("\nEC2 stop command sent successfully.")
                    print("Instance will stop shortly.")
                    print("Note: This is a STOP, not terminate.")
                else:
                    print(f"\nWarning: EC2 stop command failed (exit code {result.returncode})")
                    if result.stderr:
                        print(f"Error: {result.stderr.strip()}")
        else:
            print("\nWarning: Instance ID not found. Skipping shutdown.")
            print("Manual shutdown command:")
            print("  aws ec2 stop-instances --instance-ids <instance-id>")

    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout contacting EC2 metadata service.")
    except Exception as e:
        print(f"\nWarning: Error during EC2 shutdown: {e}")
else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("Set SHUTDOWN_EC2 = True after reviewing outputs and confirming S3 sync.")

print("\n" + "=" * 80)
print("Results review workflow complete.")
print("=" * 80)